In [1]:
from datasets import load_dataset
dataset = load_dataset("jxie/coco_captions", split="train", streaming=True)

/Users/tls/qml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from coco_processing import stream2df
subset = list(dataset.take(1000))
df_coco = stream2df(subset)
df_coco.head()

,cocoid,filename,caption
0,522418,COCO_val2014_000000522418.jpg,A woman wearing a net on her head cutting a ca...
1,522418,COCO_val2014_000000522418.jpg,A woman cutting a large white sheet cake.
2,522418,COCO_val2014_000000522418.jpg,A woman wearing a hair net cutting a large she...
3,522418,COCO_val2014_000000522418.jpg,there is a woman that is cutting a white cake
4,522418,COCO_val2014_000000522418.jpg,A woman marking a cake with the back of a chef...


In [7]:
from coco_processing import lemmatise_df
df_aug = lemmatise_df(df_coco, mode="augment")
df_aug.head()

/Users/tls/qml/lib/python3.12/site-packages/lemminflect/utils/DataContainer.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dc.__dict__ = pickle.load(f)


,cocoid,filename,caption,lemmatised
0,522418,COCO_val2014_000000522418.jpg,A woman wearing a net on her head cutting a ca...,A woman wears a net on her head cutting a cake.
1,522418,COCO_val2014_000000522418.jpg,A woman cutting a large white sheet cake.,A woman cuts a large white sheet cake.
2,522418,COCO_val2014_000000522418.jpg,A woman wearing a hair net cutting a large she...,A woman wears a hair net cutting a large sheet...
3,522418,COCO_val2014_000000522418.jpg,there is a woman that is cutting a white cake,There is a woman that is cutting a white cake.
4,522418,COCO_val2014_000000522418.jpg,A woman marking a cake with the back of a chef...,A woman marks a cake with the back of a chefs ...


In [8]:
from lambeq import BobcatParser
parser_path = '/Users/tls/Desktop/Work/COMP0267/assignment_5/COMP0267_CW/bobcat'
ccg_parser = BobcatParser(model_name_or_path=parser_path, cache_dir=parser_path)

from coco_processing import get_trees_mscoco
df_trees = get_trees_mscoco(df_aug, 'lemmatised', ccg_parser)

Error processing tree 36: Non-complex types do not have a result
Error parsing sentence 385


In [9]:
from coco_processing import tree2tn
einsum_arr = tree2tn(df_trees, labels=['lemmatised_tree'], simplify=False)

100%|██████████| 998/998 [00:00<00:00, 22688.63it/s]


In [14]:
from modules.tensor_network import analyse_einsum
from modules.quantum import IQPAnsatz, tn2qc_svo
from modules.tensor_network import tn_metadata
obmap = {'n': 2, 'p': 2, 's': 2, 'out': 3}
nl = 2
ansatz = IQPAnsatz(obmap, nl)

qc_arr = tn2qc_svo(einsum_arr, ansatz)
info = tn_metadata(qc_arr)
print("Qubit Count, Gate Count, Circuit Depth, Max Width")
print(f"Average: {info['avg']}")

100%|██████████| 998/998 [00:11<00:00, 90.20it/s] 

Qubit Count, Gate Count, Circuit Depth, Max Width
Average: (46, 320, 25, 7)
